In [ ]:
import os
import time
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from googleapiclient.discovery import build

In [ ]:
load_dotenv()

In [ ]:
api_key = os.getenv("youtube_api_key")
youtube = build("youtube", "v3", developerKey=api_key)

In [ ]:
SEARCH_QUERIES = [
    "makeup tutorial #shorts",
    "grwm makeup #shorts",
    "drugstore makeup #shorts",
    "makeup transformation #shorts",
    "no makeup makeup look #shorts",
    "glam makeup #shorts",
    "makeup dupe #shorts",
    "everyday makeup #shorts",
    "soft glam makeup #shorts",
    "viral makeup trend #shorts",
]

In [ ]:
MAX_RESULTS_PER_QUERY = 50
MAX_PAGES_PER_QUERY   = 3

In [ ]:
OUTPUT_FILE = "data/makeup_shorts_raw.csv"

In [ ]:
def search_shorts(query, max_results=50, page_token=None):
    """Search YouTube for Shorts matching a query."""
    request = youtube.search().list(
        part="snippet",
        q=query,
        type="video",
        videoDuration="short",
        order="relevance",
        maxResults=max_results,
        pageToken=page_token
    )
    return request.execute()


In [ ]:
def get_video_details(video_ids):
    """
    Given a list of video IDs, fetch full stats + content details.
    Returns raw items list from the API.
    """
    # API allows max 50 IDs per call
    request = youtube.videos().list(
        part="statistics,contentDetails,snippet",
        id=",".join(video_ids)
    )
    response = request.execute()
    return response.get("items", [])

In [ ]:
def get_channel_details(channel_ids):
    """
    Given a list of channel IDs, fetch subscriber counts.
    Returns a dict: {channel_id: subscriber_count}
    """
    unique_ids = list(set(channel_ids))
    request = youtube.channels().list(
        part="statistics",
        id=",".join(unique_ids)
    )
    response = request.execute()

    channel_data = {}
    for item in response.get("items", []):
        cid = item["id"]
        stats = item.get("statistics", {})
        channel_data[cid] = {
            "subscriber_count": int(stats.get("subscriberCount", 0)),
            "total_channel_views": int(stats.get("viewCount", 0)),
            "total_videos": int(stats.get("videoCount", 0)),
        }
    return channel_data

In [ ]:
def parse_duration_to_seconds(duration_str):
    """
    Convert ISO 8601 duration (e.g. 'PT58S', 'PT1M30S') to seconds.
    Used to filter true Shorts (<= 60 seconds).
    """
    import re
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    match = re.match(pattern, duration_str)
    if not match:
        return 0
    hours   = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)
    return hours * 3600 + minutes * 60 + seconds

In [ ]:
def collect_all_shorts():
    """
    Main collection loop.
    Iterates over all search queries, paginates, fetches details,
    and returns a flat list of video records.
    """
    all_records = []
    seen_video_ids = set()  # deduplicate across queries

    for query in SEARCH_QUERIES:
        print(f"\n🔍 Searching: '{query}'")
        page_token = None

        for page_num in range(MAX_PAGES_PER_QUERY):
            try:
                search_response = search_shorts(query, MAX_RESULTS_PER_QUERY, page_token)
            except Exception as e:
                print(f"  ⚠️  Search error on page {page_num+1}: {e}")
                break

            items = search_response.get("items", [])
            if not items:
                break

            # collect new video IDs only
            video_ids = []
            for item in items:
                vid_id = item["id"].get("videoId")
                if vid_id and vid_id not in seen_video_ids:
                    video_ids.append(vid_id)
                    seen_video_ids.add(vid_id)

            if not video_ids:
                break

            # fetch full video stats
            video_details = get_video_details(video_ids)

            # collect channel IDs to batch-fetch subscriber counts
            channel_ids = [v["snippet"]["channelId"] for v in video_details]
            channel_info = get_channel_details(channel_ids)

            # parse each video into a flat record
            for video in video_details:
                snippet  = video.get("snippet", {})
                stats    = video.get("statistics", {})
                content  = video.get("contentDetails", {})

                duration_secs = parse_duration_to_seconds(
                    content.get("duration", "PT0S")
                )

                # ── filter: only true Shorts (<= 60 seconds) ──
                if duration_secs > 60:
                    continue

                vid_id     = video["id"]
                channel_id = snippet.get("channelId", "")
                ch_info    = channel_info.get(channel_id, {})

                views    = int(stats.get("viewCount",    0))
                likes    = int(stats.get("likeCount",    0))
                comments = int(stats.get("commentCount", 0))

                # days since upload
                published_at = snippet.get("publishedAt", "")
                try:
                    pub_date = datetime.strptime(published_at, "%Y-%m-%dT%H:%M:%SZ")
                    days_since_upload = (datetime.utcnow() - pub_date).days
                except:
                    days_since_upload = None
                    pub_date = None

                record = {
                    # identifiers
                    "video_id":            vid_id,
                    "channel_id":          channel_id,
                    "search_query":        query,

                    # content info
                    "title":               snippet.get("title", ""),
                    "description":         snippet.get("description", "")[:300],
                    "tags":                "|".join(snippet.get("tags", [])),
                    "published_at":        published_at,
                    "duration_seconds":    duration_secs,

                    # engagement metrics
                    "views":               views,
                    "likes":               likes,
                    "comments":            comments,

                    # derived features
                    "engagement_rate":     round((likes + comments) / views, 4) if views > 0 else 0,
                    "like_rate":           round(likes / views, 4) if views > 0 else 0,
                    "comment_rate":        round(comments / views, 4) if views > 0 else 0,
                    "days_since_upload":   days_since_upload,
                    "view_velocity":       round(views / days_since_upload, 1) if days_since_upload and days_since_upload > 0 else views,

                    # channel info
                    "subscriber_count":    ch_info.get("subscriber_count", 0),
                    "total_channel_views": ch_info.get("total_channel_views", 0),
                    "total_videos":        ch_info.get("total_videos", 0),

                    # url for reference
                    "url":                 f"https://youtube.com/shorts/{vid_id}",
                }

                all_records.append(record)

            print(f" Page {page_num+1}: collected {len(video_ids)} videos | Total so far: {len(all_records)}")

            # next page
            page_token = search_response.get("nextPageToken")
            if not page_token:
                break

            time.sleep(0.5)  # be nice to the API

        time.sleep(1)  # pause between queries

    return all_records

## PHASE 1 - Initial Data Collection

In [ ]:
if __name__ == "__main__":
    print("Starting Makeup Shorts Data Collection...")
    print(f"   Queries: {len(SEARCH_QUERIES)}")
    print(f"   Max videos per query: {MAX_RESULTS_PER_QUERY * MAX_PAGES_PER_QUERY}")
    print(f"   Estimated total: ~{len(SEARCH_QUERIES) * MAX_RESULTS_PER_QUERY * MAX_PAGES_PER_QUERY} videos\n")

    os.makedirs("data", exist_ok=True)

    records = collect_all_shorts()

    if records:
        df = pd.DataFrame(records)
        df.to_csv(OUTPUT_FILE, index=False)

        print(f"\nDone! Collected {len(df)} unique Shorts")
        print(f"   Saved to: {OUTPUT_FILE}")
        print(f"\nQuick Stats:")
        print(f"   Avg views:          {df['views'].mean():,.0f}")
        print(f"   Avg engagement rate: {df['engagement_rate'].mean():.2%}")
        print(f"   Avg duration:        {df['duration_seconds'].mean():.1f}s")
        print(f"   Subscriber range:    {df['subscriber_count'].min():,} – {df['subscriber_count'].max():,}")
    else:
        print("No records collected. Check your API key.")

In [ ]:
shorts_raw_df = pd.read_csv("data/makeup_shorts_raw.csv")

In [ ]:
shorts_raw_df.head(10)

## PHASE 2 - Append Additional Queries

In [ ]:
existing_ids   = set(shorts_raw_df["video_id"].tolist())
print(f"   Existing videos: {len(shorts_raw_df)}")
print(f"   Unique video IDs: {len(existing_ids)}")

In [ ]:
NEW_QUERIES = [
    "foundation routine #shorts",
    "concealer hack #shorts",
    "contour tutorial #shorts",
    "eyeshadow tutorial #shorts",
    "lip liner tutorial #shorts",
    "affordable makeup #shorts",
    "luxury makeup #shorts",
    "beginner makeup #shorts",
    "full face makeup #shorts",
    "makeup for school #shorts",
    "natural glam makeup #shorts",
    "bold makeup look #shorts",
    "monochromatic makeup #shorts",
    "makeup artist #shorts",
    "get ready with me glam #shorts",
]

In [ ]:
def collect_new_shorts(existing_ids):
    all_records  = []
    seen_new_ids = set()

    for query in NEW_QUERIES:
        print(f"\n Searching: '{query}'")
        page_token = None

        for page_num in range(MAX_PAGES_PER_QUERY):
            try:
                search_response = search_shorts(query, MAX_RESULTS_PER_QUERY, page_token)
            except Exception as e:
                print(f"  ⚠️  Search error on page {page_num+1}: {e}")
                break

            items = search_response.get("items", [])
            if not items:
                break

            # filter out already existing + already seen this run
            video_ids = []
            for item in items:
                vid_id = item["id"].get("videoId")
                if vid_id and vid_id not in existing_ids and vid_id not in seen_new_ids:
                    video_ids.append(vid_id)
                    seen_new_ids.add(vid_id)

            if not video_ids:
                print(f"  ⏭️  Page {page_num+1}: all duplicates, skipping")
                break

            video_details = get_video_details(video_ids)
            channel_ids   = [v["snippet"]["channelId"] for v in video_details]
            channel_info  = get_channel_details(channel_ids)

            for video in video_details:
                snippet  = video.get("snippet", {})
                stats    = video.get("statistics", {})
                content  = video.get("contentDetails", {})

                duration_secs = parse_duration_to_seconds(content.get("duration", "PT0S"))
                if duration_secs > 60:
                    continue

                vid_id     = video["id"]
                channel_id = snippet.get("channelId", "")
                ch_info    = channel_info.get(channel_id, {})

                views    = int(stats.get("viewCount",    0))
                likes    = int(stats.get("likeCount",    0))
                comments = int(stats.get("commentCount", 0))

                published_at = snippet.get("publishedAt", "")
                try:
                    pub_date          = datetime.strptime(published_at, "%Y-%m-%dT%H:%M:%SZ")
                    days_since_upload = (datetime.utcnow() - pub_date).days
                except:
                    days_since_upload = None

                record = {
                    "video_id":            vid_id,
                    "channel_id":          channel_id,
                    "search_query":        query,
                    "title":               snippet.get("title", ""),
                    "description":         snippet.get("description", "")[:300],
                    "tags":                "|".join(snippet.get("tags", [])),
                    "published_at":        published_at,
                    "duration_seconds":    duration_secs,
                    "views":               views,
                    "likes":               likes,
                    "comments":            comments,
                    "engagement_rate":     round((likes + comments) / views, 4) if views > 0 else 0,
                    "like_rate":           round(likes / views, 4) if views > 0 else 0,
                    "comment_rate":        round(comments / views, 4) if views > 0 else 0,
                    "days_since_upload":   days_since_upload,
                    "view_velocity":       round(views / days_since_upload, 1) if days_since_upload and days_since_upload > 0 else views,
                    "subscriber_count":    ch_info.get("subscriber_count", 0),
                    "total_channel_views": ch_info.get("total_channel_views", 0),
                    "total_videos":        ch_info.get("total_videos", 0),
                    "url":                 f"https://youtube.com/shorts/{vid_id}",
                }
                all_records.append(record)

            print(f"  Page {page_num+1}: +{len(video_ids)} new videos | New total: {len(all_records)}")

            page_token = search_response.get("nextPageToken")
            if not page_token:
                break

            time.sleep(0.5)

        time.sleep(1)

    return all_records

In [ ]:
print(f"Pulling new data from {len(NEW_QUERIES)} additional queries...")
new_records = collect_new_shorts(existing_ids)

if new_records:
    new_df     = pd.DataFrame(new_records)
    combined_df = pd.concat([shorts_raw_df, new_df], ignore_index=True)
    combined_df.drop_duplicates(subset="video_id", inplace=True)
    combined_df.to_csv(OUTPUT_FILE, index=False)

    print(f"Done!")
    print(f"   Previous count:  {len(shorts_raw_df):,}")
    print(f"   New videos added: {len(new_df):,}")
    print(f"   Total now:        {len(combined_df):,}")
    print(f"   Saved to: {OUTPUT_FILE}")
else:
    print("No new records collected. Check your API key or quota.")

## Save Final Dataset

In [ ]:
os.makedirs("data", exist_ok=True)
combined_df = pd.concat([shorts_raw_df, pd.DataFrame(new_records)], ignore_index=True)
combined_df.drop_duplicates(subset="video_id", inplace=True)
combined_df.to_csv("data/makeup_shorts_raw.csv", index=False)
print(f"Saved {len(combined_df)} total records")